# Multimodal Acquisition, Calibration, And Manifests

PyTex now has a stable shared experiment layer. This notebook shows how acquisition
semantics, calibration state, quality metadata, and machine-readable manifests fit
together.


In [1]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [2]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

acquisition = AcquisitionGeometry(
    specimen_frame=specimen,
    modality="ebsd",
    map_frame=map_frame,
    specimen_to_map=FrameTransform(
        source=specimen,
        target=map_frame,
        rotation_matrix=np.eye(3),
    ),
    calibration_record=CalibrationRecord(
        source="stage-fit",
        status="calibrated",
        residual_error=0.1,
    ),
    measurement_quality=MeasurementQuality(
        confidence=0.95,
        valid_fraction=0.99,
        uncertainty={"tilt_deg": 0.1},
    ),
)

experiment = ExperimentManifest.from_acquisition_geometry(
    acquisition,
    source_system="pytex",
    phase=phase,
    referenced_files=("scan.ang",),
)
print(experiment.to_dict()["schema_id"])
print(experiment.to_dict()["modality"])


pytex.experiment_manifest
ebsd


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:720: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1224: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


In [3]:
benchmark = BenchmarkManifest(
    benchmark_id="ebsd_regular_grid_demo",
    subsystem="ebsd",
    baseline_kind="internal_plus_mtex",
    workflows=("kam", "segmentation", "grod"),
    tolerances={"misorientation_atol_deg": 1e-6},
)

validation = ValidationManifest(
    campaign_name="texture_validation_demo",
    subsystem="texture",
    baseline_kind="mtex_plus_internal",
    status="implemented",
    reference_ids=("MTEX", "Bunge"),
    linked_benchmark_ids=(benchmark.benchmark_id,),
)

workflow = WorkflowResultManifest(
    result_id="demo_result",
    workflow_name="ebsd_pipeline",
    modality="ebsd",
    produced_by="pytex",
    input_manifest_ids=(experiment.schema_id,),
    artifact_paths=("results/demo_figure.svg",),
)

print(benchmark.to_dict()["workflows"])
print(validation.to_dict()["reference_ids"])
print(workflow.to_dict()["artifact_paths"])


['kam', 'segmentation', 'grod']
['MTEX', 'Bunge']
['results/demo_figure.svg']
